## 8. Prediccion_Resultados

Entrena un modelo que estima la probabilidad de victoria local, empate o victoria visitante,
lo registra en MLflow (Unity Catalog) y escribe `gold.match_predictions`.

- Variables previas al partido: diferencia de Elo y forma de los ultimos 8 partidos de cada
  club dentro de la temporada (puntos, diferencia de goles y diferencia de tiros a puerta).
- Validacion: para cada una de las 4 temporadas anteriores a la ultima se entrena con las
  previas y se evalua en esa temporada. Las metricas quedan en el run de MLflow.
- Modelo final: entrena con todas las temporadas menos la ultima, que queda como prueba.
- Las predicciones de entrenamiento se marcan como `entrenamiento` (dentro de muestra).

In [0]:
dbutils.widgets.removeAll()

In [0]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from pyspark.sql.functions import col
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [0]:
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema", "gold")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("experimento", "/Shared/smartdata_football/prediccion_resultados")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
esquema_source = dbutils.widgets.get("esquema_source")
experimento = dbutils.widgets.get("experimento")
nombre_modelo = f"{catalogo}.{esquema}.match_outcome_model"

In [0]:
VENTANA_FORMA = 8
REGULARIZACION = 0.01
PUNTOS = {"W": 3, "D": 1, "L": 0}
FEATURES = ["elo_diff", "home_form_pts", "away_form_pts", "home_form_gd", "away_form_gd",
            "home_form_sot", "away_form_sot"]


def construir_features(elo):
    e = elo.sort_values(["club", "match_date", "match_id"]).copy()
    e["pts"] = e["result"].map(PUNTOS)
    e["gd"] = e["goals_for"] - e["goals_against"]
    por_temporada = e.groupby(["club", "season_year"])
    for origen, destino in (("pts", "form_pts"), ("gd", "form_gd")):
        e[destino] = por_temporada[origen].transform(
            lambda s: s.shift(1).rolling(VENTANA_FORMA, min_periods=1).mean())

    club_cols = ["club", "elo_pre", "form_pts", "form_gd"]
    local = e[e["is_home"]].set_index("match_id")
    visita = e[~e["is_home"]].set_index("match_id")[club_cols]
    df = local[["match_date", "season_year", "round", "country", "league_name",
                "goals_for", "goals_against", *club_cols]] \
        .join(visita, rsuffix="_visita") \
        .rename(columns={
            "club": "home_club", "club_visita": "away_club",
            "elo_pre": "home_elo_pre", "elo_pre_visita": "away_elo_pre",
            "form_pts": "home_form_pts", "form_pts_visita": "away_form_pts",
            "form_gd": "home_form_gd", "form_gd_visita": "away_form_gd",
            "goals_for": "home_goals", "goals_against": "away_goals",
        }).reset_index()

    df["elo_diff"] = df["home_elo_pre"] - df["away_elo_pre"]
    df["actual_result"] = "Empate"
    df.loc[df["home_goals"] > df["away_goals"], "actual_result"] = "Local"
    df.loc[df["home_goals"] < df["away_goals"], "actual_result"] = "Visitante"
    return df


def construir_forma_tiros(partidos):
    lados = []
    for lado, propios, rival in (("home", "home_shots_on_target", "away_shots_on_target"),
                                 ("away", "away_shots_on_target", "home_shots_on_target")):
        lados.append(partidos[["match_id", "match_date", "season_year"]].assign(
            club=partidos[f"{lado}_club"], lado=lado,
            sot_diff=partidos[propios] - partidos[rival]))
    t = pd.concat(lados).sort_values(["club", "match_date", "match_id"])
    t["form_sot"] = t.groupby(["club", "season_year"])["sot_diff"].transform(
        lambda s: s.shift(1).rolling(VENTANA_FORMA, min_periods=1).mean())
    forma = t.pivot(index="match_id", columns="lado", values="form_sot")
    return forma.rename(columns={"home": "home_form_sot", "away": "away_form_sot"}).reset_index()

In [0]:
df_elo = spark.table(f"{catalogo}.{esquema}.club_elo").toPandas()
df_partidos = spark.table(f"{catalogo}.{esquema_source}.matches_transformed").select(
    "match_id", "match_date", "season_year", "home_club", "away_club",
    "home_shots_on_target", "away_shots_on_target"
).toPandas()
df = construir_features(df_elo).merge(construir_forma_tiros(df_partidos), on="match_id", how="left")

temporadas = sorted(int(t) for t in df["season_year"].unique())
temporada_prueba = temporadas[-1]
temporadas_validacion = temporadas[-5:-1]
es_prueba = df["season_year"] == temporada_prueba


def nuevo_modelo():
    return make_pipeline(StandardScaler(), LogisticRegression(C=REGULARIZACION, max_iter=1000))


def variables(df, entrenamiento):
    # Si una variable viene toda vacia (por ejemplo sin tiros en la fuente) se rellena con 0
    relleno = df.loc[entrenamiento, FEATURES].mean().fillna(0)
    return df[FEATURES].fillna(relleno)


X_todo = variables(df, ~es_prueba)
X_train, y_train = X_todo[~es_prueba], df.loc[~es_prueba, "actual_result"]
X_test, y_test = X_todo[es_prueba], df.loc[es_prueba, "actual_result"]

In [0]:
from databricks.sdk import WorkspaceClient

# MLflow crea el experimento, pero no la carpeta que lo contiene
WorkspaceClient().workspace.mkdirs(experimento.rsplit("/", 1)[0])

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experimento)

with mlflow.start_run(run_name=f"logreg_prueba_{temporada_prueba}"):
    validacion = []
    for t in temporadas_validacion:
        entrenamiento = df["season_year"] < t
        evaluacion = df["season_year"] == t
        X = variables(df, entrenamiento)
        m = nuevo_modelo().fit(X[entrenamiento], df.loc[entrenamiento, "actual_result"])
        y = df.loc[evaluacion, "actual_result"]
        validacion.append({
            "temporada": t,
            "log_loss": log_loss(y, m.predict_proba(X[evaluacion]), labels=m.classes_),
            "accuracy": accuracy_score(y, m.predict(X[evaluacion])),
        })
        mlflow.log_metric("log_loss_validacion", validacion[-1]["log_loss"], step=t)
        mlflow.log_metric("accuracy_validacion", validacion[-1]["accuracy"], step=t)

    modelo = nuevo_modelo().fit(X_train, y_train)
    proba_test = modelo.predict_proba(X_test)
    pred_test = modelo.predict(X_test)
    metricas = {
        "log_loss_validacion_media": float(pd.DataFrame(validacion)["log_loss"].mean()),
        "accuracy_validacion_media": float(pd.DataFrame(validacion)["accuracy"].mean()),
        "accuracy_prueba": accuracy_score(y_test, pred_test),
        "f1_macro_prueba": f1_score(y_test, pred_test, average="macro"),
        "log_loss_prueba": log_loss(y_test, proba_test, labels=modelo.classes_),
        "accuracy_base_siempre_local": float((y_test == "Local").mean()),
    }
    mlflow.log_params({"modelo": "LogisticRegression", "features": ",".join(FEATURES),
                       "ventana_forma": VENTANA_FORMA, "C": REGULARIZACION, "temporada_prueba": temporada_prueba,
                       "temporadas_validacion": ",".join(map(str, temporadas_validacion)),
                       "partidos_entrenamiento": len(X_train), "partidos_prueba": len(X_test)})
    mlflow.log_metrics(metricas)
    mlflow.sklearn.log_model(
        modelo, "modelo",
        signature=infer_signature(X_train, modelo.predict(X_train)),
        input_example=X_train.head(5),
        registered_model_name=nombre_modelo,
    )

print(pd.DataFrame(validacion))
print(metricas)

In [0]:
clases = list(modelo.classes_)
proba = modelo.predict_proba(X_todo)
df["prob_home"] = proba[:, clases.index("Local")]
df["prob_draw"] = proba[:, clases.index("Empate")]
df["prob_away"] = proba[:, clases.index("Visitante")]
df["predicted_result"] = modelo.predict(X_todo)
df["is_correct"] = (df["predicted_result"] == df["actual_result"]).astype(int)
df["split"] = es_prueba.map({True: "prueba", False: "entrenamiento"})

In [0]:
TIPOS_PRED = [
    ("match_id", "int"), ("match_date", "date"), ("season_year", "int"), ("round", "int"),
    ("country", "string"), ("league_name", "string"), ("home_club", "string"), ("away_club", "string"),
    ("home_elo_pre", "double"), ("away_elo_pre", "double"), ("prob_home", "double"),
    ("prob_draw", "double"), ("prob_away", "double"), ("predicted_result", "string"),
    ("actual_result", "string"), ("is_correct", "int"), ("split", "string"),
]

df_pred = spark.createDataFrame(df[[c for c, _ in TIPOS_PRED]]) \
    .select([col(c).cast(t).alias(c) for c, t in TIPOS_PRED])
df_pred.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.match_predictions")